# 15. 3Sum

[Problem](https://leetcode.com/problems/3sum/) · difficulty: medium

Three approaches, 1474 ms to 327 ms on the judge. The first gap is a *correctness* decision —
which element you pin decides whether duplicate triplets can be avoided at all. The second is a
measurement question that the complexity bound gets backwards.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0015-3sum'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Pins | Dedupe |
|---|---|---|---|---|
| `SolutionMiddlePivotSet` | O(n²) | O(t) | middle element | a set, afterwards |
| `SolutionTwoPointersPruned` | O(n²) | O(1) | smallest element | skip equal values |
| `SolutionBisectJump` | O(n² log n) | O(1) | smallest element | skip equal values |

`t` is the size of the answer, itself up to O(n²).


## Why pinning the middle forces a set

Sorted input makes the two-pointer move legal either way. But with the **middle** element
pinned, the outer pointers roam the whole array, and the same triplet is reachable from more
than one pivot — duplicates appear even when every value is distinct.

Pin the **smallest** element instead and each triplet has exactly one pivot that can produce
it: its own minimum. Then the only duplicates left come from repeated values, which skipping
equal neighbours handles.

The trace counts how often each pivot choice reaches the same triplet.


In [ ]:
from collections import Counter

def emitted_pinning_middle(nums):
    """Every triplet the middle-pivot scan hits, before the set removes duplicates."""
    nums = sorted(nums)
    hits = Counter()
    for pivot in range(1, len(nums) - 1):
        left, right = 0, len(nums) - 1
        while left < pivot < right:
            total = nums[left] + nums[pivot] + nums[right]
            if total == 0:
                hits[(nums[left], nums[pivot], nums[right])] += 1
            if total < 0:
                left += 1
            else:
                right -= 1
    return hits


def emitted_pinning_smallest(nums):
    """The same, for the scan that pins the smallest element and skips equal values."""
    nums = sorted(nums)
    hits, n = Counter(), len(nums)
    for i in range(n - 2):
        if i and nums[i] == nums[i - 1]:
            continue
        left, right, target = i + 1, n - 1, -nums[i]
        while left < right:
            total = nums[left] + nums[right]
            if total == target:
                hits[(nums[i], nums[left], nums[right])] += 1
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1
                while left < right and nums[right] == nums[right + 1]:
                    right -= 1
            elif total < target:
                left += 1
            else:
                right -= 1
    return hits


NUMS = [-4, -2, -2, 0, 2, 2, 4]
print('pinning the middle, times each triplet is hit:')
for triplet, count in sorted(emitted_pinning_middle(NUMS).items()):
    print(f'  {triplet}  {count}x')
print('\npinning the smallest, times each triplet is hit:')
for triplet, count in sorted(emitted_pinning_smallest(NUMS).items()):
    print(f'  {triplet}  {count}x')


Pinning the middle hits triplets more than once even after the scan skips equal neighbours,
so the set is doing real work — remove it and the answer contains repeats. Pinning the
smallest hits each triplet exactly once, which is why local skipping is enough there and the
O(t) memory disappears.


## Where the time goes

Four shapes at the constraint limit of n = 3000. The first has values spread across the full
±100000 range, so duplicates are rare; the others concentrate them.


In [ ]:
import random, time

random.seed(3)
SHAPES = {
    'wide range': [random.randint(-100000, 100000) for _ in range(3000)],
    'values in ±30': [random.randint(-30, 30) for _ in range(3000)],
    'all zeros': [0] * 3000,
    'all positive': list(range(1, 1501)),
}

def timed(solution, nums, runs=3):
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        solution().threeSum(list(nums))
        samples.append((time.perf_counter() - start) * 1e3)
    return min(samples)

print(f"{'approach':<28}" + ''.join(f'{name:>16}' for name in SHAPES))
for solution in solutions:
    row = ''.join(f'{timed(solution, nums):13.1f} ms' for nums in SHAPES.values())
    print(f'{solution.__name__:<28}{row}')


Two readings:

- `SolutionMiddlePivotSet` loses everywhere, and worst where the answer is large — its set grows
  with the number of triplets while the others hold three indices.
- **`SolutionBisectJump` only beats the plain walk when values repeat.** On the wide-range input
  the two are within a percent of each other: a jump that lands one index along still paid
  O(log n) for it. Its advertised bound, O(n² log n), is *worse* than the plain walk's O(n²) —
  the win is real but it comes from crossing runs of equal values in one move, not from the
  asymptotics.


## The bug worth remembering

A dedup skip compares against **where the pointer came from**, not where it is going. The first
version of the bisect approach read, after `left += 1`:

```python
while left < right and nums[left] == nums[left + 1]:   # wrong
    left += 1
```

which skips a value that was never reported. It passed the examples and failed only on
duplicate-heavy input — 167 of 4000 random arrays.


In [ ]:
import random

def dedup_variant(nums, forward):
    nums = sorted(nums)
    triplets, n = [], len(nums)
    for i in range(n - 2):
        if i and nums[i] == nums[i - 1]:
            continue
        left, right, target = i + 1, n - 1, -nums[i]
        while left < right:
            total = nums[left] + nums[right]
            if total == target:
                triplets.append([nums[i], nums[left], nums[right]])
                left += 1
                right -= 1
                if forward:
                    while left < right and nums[left] == nums[left + 1]:
                        left += 1
                else:
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
            elif total < target:
                left += 1
            else:
                right -= 1
    return triplets

def norm(triplets):
    return sorted(sorted(t) for t in triplets)

random.seed(7)
wrong = 0
first = None
for _ in range(4000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(3, 9))]
    want = norm(solutions[1]().threeSum(list(nums)))
    if norm(dedup_variant(list(nums), forward=True)) != want:
        wrong += 1
        first = first or nums
print(f'looking forward : {wrong} / 4000 arrays wrong, e.g. {first}')

wrong = sum(norm(dedup_variant(list(nums), forward=False)) != norm(solutions[1]().threeSum(list(nums)))
            for nums in [[random.randint(-6, 6) for _ in range(random.randint(3, 9))] for _ in range(4000)])
print(f'looking back    : {wrong} / 4000 arrays wrong')


## Takeaway

- Pick what to pin before optimizing anything else. Pinning the smallest element is what makes
  local dedup sufficient and drops the O(t) set — a 3.6x difference on the judge.
- A worse complexity bound can still be the faster program. `SolutionBisectJump` is O(n² log n)
  against O(n²) and wins by 25% on LeetCode, because its tests repeat values and a jump crosses
  a whole run.
- Duplicate-handling bugs hide from the examples. The problem's own three cases pass with the
  skip written backwards; only randomized comparison against a reference finds it.
